In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
np.random.seed(1)  # để kết quả tái lập được

# 1. TẠO DỮ LIỆU GIẢ LẬP (MOCK DATA) CHO NHIỀU NGÀY, MỖI NGÀY 24 GIỜ
N_DAYS = 30
HOURS_PER_DAY = 24
total_hours = N_DAYS * HOURS_PER_DAY
hours = np.arange(total_hours) # sinh ra 1 tập dữ liệu từ 0 -> 719
hour_of_day = hours % HOURS_PER_DAY

# --- FIX #1: Nhiệt độ dùng 2 hài hòa (harmonics) để đáy ~5h, đỉnh ~14h --  -
# Một cosine đơn không thể cho đáy và đỉnh cách nhau 9h (chỉ cách được đúng 12h).
# Cộng thêm hài bậc 2 sẽ "kéo méo" chu kỳ để mô phỏng đúng hơn thực tế.
phase = 2 * np.pi * (hour_of_day - 5) / 24
temperature = 25 - 5 * np.cos(phase) - 1.2 * np.cos(2 * phase - np.pi / 2)
temperature += np.random.normal(0, 0.4, size=total_hours)

# Độ ẩm không khí: ngược chiều với nhiệt độ (giữ nguyên logic, đồng bộ pha)
humidity = 80 + 15 * np.cos(phase) + 3.6 * np.cos(2 * phase - np.pi / 2)
humidity += np.random.normal(0, 1.0, size=total_hours)
humidity = np.clip(humidity, 0, 100)

# Ánh sáng (Lux): chỉ có mặt trời từ 6h đến 18h, đỉnh lúc 12h trưa (không đổi, đã đúng)
light_lux = np.where(
    (hour_of_day >= 6) & (hour_of_day <= 18),
    50000 * np.sin(np.pi * (hour_of_day - 6) / 12),
    0
)
light_lux = light_lux + np.random.normal(0, 300, size=total_hours)
light_lux = np.clip(light_lux, 0, None)

# Độ ẩm đất (giữ nguyên logic vòng lặp, không đổi)
soil_moisture = np.zeros(total_hours)
soil_moisture[0] = 60
for i in range(1, total_hours):
    if hour_of_day[i] == 0:
        recovery = np.random.uniform(15, 25)
        soil_moisture[i] = min(70, soil_moisture[i - 1] + recovery)
    else:
        evap = 0.3 + (light_lux[i] / 10000) * 0.5
        soil_moisture[i] = soil_moisture[i - 1] - evap
soil_moisture = np.clip(soil_moisture, 0, 100)
soil_moisture += np.random.normal(0, 0.3, size=total_hours)

df = pd.DataFrame({
    'Hour': hours,
    'Hour_of_day': hour_of_day,
    'Temperature_C': temperature,
    'Humidity_%': humidity,
    'Light_Lux': light_lux,
    'Soil_Moisture_%': soil_moisture
})

print("--- DỮ LIỆU CẢM BIẾN MÔ PHỎNG (5 GIỜ ĐẦU) ---")
print(df.head())
print(f"\nTổng số mẫu: {len(df)} giờ ({N_DAYS} ngày)")

# Kiểm tra nhanh: giờ nào nhiệt độ đạt đỉnh?
peak_hour = df.groupby('Hour_of_day')['Temperature_C'].mean().idxmax()
print(f"Giờ nhiệt độ trung bình cao nhất: {peak_hour}h (mục tiêu ~14h)")

# 2. CHUẨN HÓA DỮ LIỆU (MIN-MAX SCALING) CHO MẠNG LSTM
train_size = int(len(df) * 0.8)
feature_cols = ['Temperature_C', 'Humidity_%', 'Light_Lux', 'Soil_Moisture_%']
scaler = MinMaxScaler()
scaler.fit(df[feature_cols].iloc[:train_size])  
scaled_data = scaler.transform(df[feature_cols])
scaled_df = pd.DataFrame(scaled_data, columns=['Temp_Scaled', 'Humid_Scaled', 'Light_Scaled', 'Soil_Scaled'])

print("\n--- DỮ LIỆU SAU KHI CHUẨN HÓA [0, 1] (FLOAT32, DÙNG ĐỂ TRAIN) ---")
print(scaled_df.head())

# --- FIX #3: Tạo sequence (sliding window) cho LSTM ---
# Sửa lại hàm tạo sequence để nhãn y chỉ lấy duy nhất cột Độ ẩm đất (Index 3)
def create_sequences(data: np.ndarray, target_col_idx: int = 3, window_size: int = 24, horizon: int = 6):
    """
    - data: Mảng scaled_data (shape: N, 4)
    - target_col_idx: Vị trí cột Độ ẩm đất (mặc định = 3)
    - window_size: 24 giờ quá khứ
    - horizon: Dự báo trước 6 giờ (hoặc 12 giờ)
    """
    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i : i + window_size])                           # Shape: [24, 4]
        y.append(data[i + window_size + horizon - 1, target_col_idx]) # Shape: [1]
    return np.array(X), np.array(y)

# Tạo tập dữ liệu chuẩn theo Spec: Input [N, 24, 4], Output [N, 1]
WINDOW_SIZE = 24
HORIZON = 6  # Dự báo độ ẩm đất sau 6 giờ tới

X_all, y_all = create_sequences(scaled_data, target_col_idx=3, window_size=WINDOW_SIZE, horizon=HORIZON)
y_all = np.expand_dims(y_all, axis=-1)  # Đưa về đúng shape [N, 1]

split_idx = train_size - WINDOW_SIZE
X_train, y_train = X_all[:split_idx], y_all[:split_idx]
X_test, y_test = X_all[split_idx:], y_all[split_idx:]

print(f"X_train shape: {X_train.shape}")  # Kết quả: (samples, 24, 4)
print(f"y_train shape: {y_train.shape}")  # Kết quả: (samples, 1) -> Khớp 100% Spec

# 3. LƯỢNG TỬ HÓA INT8 (CHỈ KHI DEPLOY LÊN VI ĐIỀU KHIỂN)
# --- FIX #2: lưu lại scale & zero_point để dequantize khi đọc output từ TFLite Micro ---
QUANT_SCALE = 1.0 / 255.0   # tương ứng: giá trị thực = (int8_value + 128) * QUANT_SCALE (trong [0,1])
QUANT_ZERO_POINT = -128

def quantize_to_int8(data_float_0_1: np.ndarray) -> np.ndarray:
    """Chuyển dữ liệu float trong khoảng [0,1] sang int8 [-128, 127]."""
    scaled_255 = data_float_0_1 * 255.0
    return np.round(scaled_255 - 128).astype(np.int8)

def dequantize_from_int8(data_int8: np.ndarray) -> np.ndarray:
    """Chuyển ngược int8 về float [0,1] — dùng khi đọc output model trên vi điều khiển."""
    return (data_int8.astype(np.float32) - QUANT_ZERO_POINT) * QUANT_SCALE

int8_data = quantize_to_int8(scaled_data)
int8_df = pd.DataFrame(int8_data, columns=['Temp_int8', 'Humid_int8', 'Light_int8', 'Soil_int8'])

print("\n--- DỮ LIỆU SAU KHI LƯỢNG TỬ HÓA INT8 (DÙNG CHO DEPLOY TFLITE MICRO) ---")
print(int8_df.head())
print(f"Scale = {QUANT_SCALE}, Zero point = {QUANT_ZERO_POINT}  (lưu lại 2 giá trị này khi export .tflite)")

--- DỮ LIỆU CẢM BIẾN MÔ PHỎNG (5 GIỜ ĐẦU) ---
   Hour  Hour_of_day  Temperature_C  Humidity_%   Light_Lux  Soil_Moisture_%
0     0            0      24.504590   82.390087  198.864381        59.830890
1     1            1      23.483925   82.672140  352.042157        59.657488
2     2            2      22.923542   85.658416   54.306468        59.319458
3     3            3      22.318315   90.615954    0.000000        58.710040
4     4            4      20.676710   92.859753  119.906386        58.908046

Tổng số mẫu: 720 giờ (30 ngày)
Giờ nhiệt độ trung bình cao nhất: 15h (mục tiêu ~14h)

--- DỮ LIỆU SAU KHI CHUẨN HÓA [0, 1] (FLOAT32, DÙNG ĐỂ TRAIN) ---
   Temp_Scaled  Humid_Scaled  Light_Scaled  Soil_Scaled
0     0.446184      0.557263      0.003939     1.000000
1     0.365851      0.564973      0.006973     0.997140
2     0.321745      0.646598      0.001076     0.991566
3     0.274110      0.782103      0.000000     0.981515
4     0.144906      0.843434      0.002375     0.984781

--